## 01. 앙상블(Ensemble)

앙상블은 여러 모델의 예측을 모아 하나의 최종 예측을 만드는 방법이다.

한 모델의 판단만 사용하는 것이 아니라, 여러 모델의 장점을 함께 활용해 더 안정적인 결과를 기대할 수 있음.

### 01-01. 배우는 이유

- 머신러닝 모델 하나는 데이터의 일부 패턴은 잘 잡지만, 동시에 자신만의 약점도 가질 수 있음.
- 서로 다른 모델을 함께 사용하면 한 모델의 실수를 다른 모델이 보완할 수 있음.
- 실무에서는 단일 모델보다 앙상블 모델이 더 안정적인 성능을 보이는 경우가 많음.

### 01-02. 어디서 사용하는가?

- 분류: 고객 이탈 예측, 질병 여부 예측, 불량품 판정처럼 여러 class 중 하나를 고르는 문제.
- 회귀: 집값, 매출, 수요량처럼 숫자를 예측하는 문제.
- 대회/실무 모델링: 여러 모델의 결과를 결합해 성능을 안정화하거나 마지막 성능을 끌어올릴 때 사용함.

### 01-03. 앙상블의 대표 방식

**Voting(투표)**
- 서로 다른 모델 여러 개를 학습함.
- 각 모델의 예측을 투표 또는 평균으로 합침.
- 앙상블 모델의 가장 직관적인 방식임.

**Bagging(Bootstrap Aggregating)**
- 같은 종류의 모델을 여러 개 만들되, 학습 데이터를 조금씩 다르게 뽑아 학습함.
- 대표 예시는 Random Forest임.
- 모델의 예측 변동을 줄이는 데 도움을 줌.

**Boosting(강화)**
- 모델을 순서대로 학습함.
- 앞 모델이 틀린 데이터를 다음 모델이 더 신경 쓰도록 만듦.
- 대표 예시는 Gradient Boosting, XGBoost, LightGBM 등임.

**Stacking(쌓기)**
- 여러 모델의 예측 결과를 다시 새로운 모델의 입력으로 사용함.
- 예측을 합치는 방법 자체를 또 다른 모델이 학습함.

### 01-04. Voting의 핵심 동작 방식

Voting은 서로 다른 모델의 예측을 **투표 또는 평균**으로 합치는 방식임.

- 분류 문제: 여러 모델이 예측한 class 또는 확률을 합쳐 최종 class를 결정함.
- 회귀 문제: 여러 모델이 예측한 숫자값을 평균내어 최종 예측값을 만듦.

### 01-05. Voting을 왜 먼저 배우는가?

**모델 하나의 실수를 줄일 수 있음**
- 한 모델이 놓치는 패턴을 다른 모델이 보완할 수 있음.
- 여러 모델의 판단을 합치면 예측이 더 안정될 수 있음.

**앙상블의 기본 아이디어를 이해하기 좋음**
- Voting은 Bagging, Boosting, Stacking보다 직관적임.
- 앙상블을 처음 설명할 때 가장 먼저 보여주기 좋음.

**모델 비교와 평가 흐름을 복습하기 좋음**
- 개별 모델 점수와 앙상블 점수를 나란히 비교함.
- 앙상블이 항상 더 좋은지 직접 확인할 수 있음.

### 01-06. Hard Voting과 Soft Voting

**Hard Voting**
- 각 모델이 예측한 최종 class를 모아 다수결로 결정함.
- 예: 세 모델의 예측이 `[악성, 양성, 양성]`이면 최종 예측은 `양성`임.

**Soft Voting**
- 각 모델이 계산한 class별 확률을 평균내어 결정함.
- 확률 정보까지 사용하므로 잘 보정된 모델들이 있을 때 유리할 수 있음.
- Soft Voting을 사용하려면 개별 모델이 `predict_proba()`를 제공해야 함.

### 01-07. Voting을 사용할 때 주의할 점

- 비슷한 모델만 여러 개 모으면 예측이 다양해지지 않아 효과가 작을 수 있음.
- 성능이 낮은 모델을 무조건 섞으면 오히려 결과가 나빠질 수 있음.
- Voting 결과가 특정 개별 모델 점수와 같게 나와도, 그 모델을 선택한 것이 아니라 샘플별 투표 결과의 전체 정답 수가 우연히 같아진 것임.


## 02. 실습 환경 준비

- `VotingClassifier`: 여러 분류 모델의 예측을 hard/soft voting으로 결합함.
- `VotingRegressor`: 여러 회귀 모델의 예측값을 평균내어 결합함.
- `LogisticRegression`, `KNeighborsClassifier`, `DecisionTreeClassifier`: Voting에 넣을 개별 분류 모델임.
- `LinearRegression`, `KNeighborsRegressor`, `DecisionTreeRegressor`: VotingRegressor에 넣을 개별 회귀 모델임.
- `StandardScaler`: 거리 기반 모델과 선형 모델이 안정적으로 학습되도록 feature 스케일을 맞춤.



In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import VotingClassifier, VotingRegressor
from sklearn.metrics import accuracy_score, classification_report, root_mean_squared_error, r2_score


## 03. 분류 데이터 로드와 스케일링

- Breast Cancer 데이터는 종양의 여러 수치 feature로 악성/양성을 분류하는 데이터임.
- Logistic Regression과 KNN은 feature 스케일의 영향을 받을 수 있으므로 스케일링함.
- Decision Tree는 스케일링이 필수는 아니지만, Voting 안에서 같은 입력을 사용하기 위해 스케일링된 데이터를 함께 사용함.



In [2]:
cancer = load_breast_cancer(as_frame=True)

voting_X = cancer.data
voting_y = cancer.target

print('분류 데이터:', voting_X.shape, voting_y.shape)
print('target 이름:', cancer.target_names)
display(voting_X.head())


분류 데이터: (569, 30) (569,)
target 이름: ['malignant' 'benign']


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [3]:
voting_X_train, voting_X_test, voting_y_train, voting_y_test = train_test_split(
    voting_X,
    voting_y,
    test_size=0.2,       # 전체 데이터의 20%를 최종 평가 데이터로 분리
    random_state=42,     # 수업 중 같은 분할 결과가 나오도록 난수 고정
    stratify=voting_y    # 악성/양성 비율이 train/test에 비슷하게 유지되도록 분할
)

voting_scaler = StandardScaler()

voting_X_train_scaled = voting_scaler.fit_transform(voting_X_train)
voting_X_test_scaled = voting_scaler.transform(voting_X_test)

print('학습 데이터:', voting_X_train_scaled.shape, voting_y_train.shape)
print('평가 데이터:', voting_X_test_scaled.shape, voting_y_test.shape)
print('target 이름:', cancer.target_names)

학습 데이터: (455, 30) (455,)
평가 데이터: (114, 30) (114,)
target 이름: ['malignant' 'benign']


## 04. 개별 분류 모델 준비

- Voting에 넣을 개별 모델을 먼저 준비함.
- 서로 다른 학습 방식의 모델을 섞어야 앙상블의 다양성을 확보하기 좋음.
- 여기서는 선형 모델, 거리 기반 모델, 트리 모델을 함께 사용함.



In [4]:
# LogisticRegressor() : 선형 방적식 결과를 시그모이드 함수를 이용해 예측/분류
lr_clf = LogisticRegression(max_iter=5000, random_state=42)

# KNeighborsClassifier() : 가까운 이웃 K개의 정답을 보고 다수결로 예측/분류
knn_clf = KNeighborsClassifier(n_neighbors=5)

# DecisionTreeClassifier() : feature 조건을 이용해 반복적으로 나누어 예측/분류
dt_clf = DecisionTreeClassifier(max_depth=4, random_state=42)

base_classifiers = {
    'logistic_regression': lr_clf,
    'knn': knn_clf,
    'decision_tree': dt_clf
}

print("개별 모델 목록: ", base_classifiers.keys())

개별 모델 목록:  dict_keys(['logistic_regression', 'knn', 'decision_tree'])


## 05. Hard Voting 분류

- Hard Voting은 각 모델이 예측한 class를 다수결로 합침.
- 확률을 사용하지 않고 최종 class 예측값만 사용함.
- 모델들이 서로 다른 예측을 할 때 다수결 결과가 최종 예측이 됨.



In [5]:
# VotingClassifier() : 여러 분류 모델의 예측을 모아
# 최종 class(정답)를 정하는 앙상블 모델

# estimators : 앙상블에 참여할 개별 모델 목록
# 참여시 ('이름', 모델) tuple 형태로 작성

# voting='hard' : 개별 모델의 예측을 다수결로 결정함
# - predict() 반환값 : 다수결로 정해진 결과
# - predict_proba() 미제공
hard_voting_clf = VotingClassifier(
    estimators=[
        ('lr', lr_clf),
        ('knn', knn_clf),
        ('dt', dt_clf),
    ],
    voting='hard',
)

hard_results = []   # 결과 저장용 변수
for name, model in base_classifiers.items():
    model.fit(voting_X_train_scaled, voting_y_train)

    hard_results.append({
        'model' : name,
        'train_accuracy' : model.score(voting_X_train_scaled, voting_y_train),
        'test_accuracy' : model.score(voting_X_test_scaled, voting_y_test)
    })

# 학습
hard_voting_clf.fit(voting_X_train_scaled, voting_y_train)

# hard_results에 hard_voting_clf 결과도 추가
hard_results.append({
    'model' : 'hard_voting',
    'train_accuracy' : hard_voting_clf.score(voting_X_train_scaled, voting_y_train),
    'test_accuracy' : hard_voting_clf.score(voting_X_test_scaled, voting_y_test)
})

hard_results_df = pd.DataFrame(hard_results)
hard_results_df

# 출력표를 보고 알 수 있는 것
# - test_accuracy 점수가 같더라도 우연히 예픅 성공한 데이터의 개수가 같은 것일 뿐이다
# - 앙상블을 통한 결과가 항상 개별 모델보다 무조건 좋다고 생각하면 안된다.

,model,train_accuracy,test_accuracy
0,logistic_regression,0.989011,0.982456
1,knn,0.973626,0.956140
2,decision_tree,0.986813,0.938596
3,hard_voting,0.986813,0.982456


## 06. Hard Voting 예측 결과 확인

- 같은 샘플에 대해 개별 모델들이 어떤 class를 예측했는지 비교함.
- Hard Voting의 최종 예측이 다수결로 결정된다는 점을 표로 확인함.
- `0`과 `1` 숫자는 `cancer.target_names`의 class 인덱스를 의미함.



In [6]:
# test 데이터 일부만 확인
sample_start = 50
sample_end = 100

hard_pred_df = pd.DataFrame({
    '실제 정답': voting_y_test[sample_start:sample_end],
    'lr 예측값': lr_clf.predict(voting_X_test_scaled[sample_start:sample_end]),
    'knn 예측값': knn_clf.predict(voting_X_test_scaled[sample_start:sample_end]),
    'dt 예측값' : dt_clf.predict(voting_X_test_scaled[sample_start:sample_end]),
    '투표 예측값': hard_voting_clf.predict(voting_X_test_scaled[sample_start:sample_end])
})

# display(hard_pred_df)

# classification_report()
hard_y_pred = hard_voting_clf.predict(voting_X_test_scaled)
print(classification_report(
    voting_y_test,
    hard_y_pred,
    target_names=cancer.target_names,
))

              precision    recall  f1-score   support

   malignant       1.00      0.95      0.98        42
      benign       0.97      1.00      0.99        72

    accuracy                           0.98       114
   macro avg       0.99      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114



## 07. Soft Voting 분류

- Soft Voting은 각 모델의 class별 예측 확률을 평균냄.
- 평균 확률이 가장 높은 class를 최종 예측으로 선택함.
- 개별 모델들이 `predict_proba()`를 제공해야 사용할 수 있음.



In [7]:
soft_voting_clf = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(max_iter=5000, random_state=42)),
        ('knn', KNeighborsClassifier(n_neighbors=5)),
        ('dt', DecisionTreeClassifier(max_depth=4, random_state=42)),
    ],
    voting='soft'
    # class(정답) 1의 확률이 각 모델별로 [0.4, 0.3, 0.2] 이면 평균 0.3
    # class(정답) 0의 확률이 각 모델별로 [0.6, 0.7, 0.8] 이면 평균 0.7
)

# 학습
soft_voting_clf.fit(voting_X_train_scaled, voting_y_train)

soft_results = hard_results_df.copy()

soft_results_df = pd.concat([
    soft_results,
    pd.DataFrame([{
    'model' : '투표(soft)',
    'train_accuracy' : soft_voting_clf.score(voting_X_train_scaled, voting_y_train),
    'test_accuracy'  : soft_voting_clf.score(voting_X_test_scaled, voting_y_test)
    }])
], ignore_index=True)
display(soft_results_df)
# hard/soft 결과를 비교해서 더 정확도가 높은 voting 방식 선택

,model,train_accuracy,test_accuracy
0,logistic_regression,0.989011,0.982456
1,knn,0.973626,0.956140
2,decision_tree,0.986813,0.938596
3,hard_voting,0.986813,0.982456
4,투표(soft),0.989011,0.956140


## 08. Soft Voting 확률 평균 확인

- Soft Voting은 class별 확률을 평균내는 방식임.
- 아래 표에서는 양성 class 확률을 기준으로 개별 모델 확률과 평균 확률을 비교함.
- 평균 확률이 0.5 이상이면 양성 class로 예측하는 흐름을 직관적으로 볼 수 있음.



In [8]:
sample_start = 20
sample_stop = 30

trained_lr, trained_knn, trained_dt = soft_voting_clf.estimators_

# predict_proba(): 각 class에 속할 확률을 반환함.
lr_proba = trained_lr.predict_proba(voting_X_test_scaled[sample_start:sample_stop])[:, 1]
knn_proba = trained_knn.predict_proba(voting_X_test_scaled[sample_start:sample_stop])[:, 1]
dt_proba = trained_dt.predict_proba(voting_X_test_scaled[sample_start:sample_stop])[:, 1]

# soft_voting_clf.predict_proba(): 내부 모델들의 class별 확률을 평균낸 최종 확률을 반환함.
soft_proba = soft_voting_clf.predict_proba(voting_X_test_scaled[sample_start:sample_stop])[:, 1]

soft_proba_df = pd.DataFrame({
    'actual': voting_y_test.to_numpy()[sample_start:sample_stop],
    'lr_proba_class_1': lr_proba,
    'knn_proba_class_1': knn_proba,
    'dt_proba_class_1': dt_proba,

    # manual_avg_proba: 개별 모델 확률을 직접 평균낸 값.
    # soft_voting_proba와 거의 같아야 Soft Voting의 계산 방식을 이해할 수 있음.
    'manual_avg_proba': (lr_proba + knn_proba + dt_proba) / 3,
    'soft_voting_proba': soft_proba,
    'soft_voting_pred': soft_voting_clf.predict(voting_X_test_scaled[sample_start:sample_stop])
})

display(soft_proba_df.round(3))

,actual,lr_proba_class_1,knn_proba_class_1,dt_proba_class_1,manual_avg_proba,soft_voting_proba,soft_voting_pred
0,0,0.002,0.0,0.000,0.001,0.001,0
1,0,0.052,0.0,0.000,0.017,0.017,0
2,1,1.000,1.0,0.992,0.997,0.997,1
3,1,0.999,1.0,0.992,0.997,0.997,1
4,1,0.990,1.0,0.992,0.994,0.994,1
5,1,0.867,1.0,0.000,0.622,0.622,1
6,0,0.000,0.0,0.000,0.000,0.000,0
7,1,0.990,1.0,0.992,0.994,0.994,1
8,1,0.999,1.0,0.992,0.997,0.997,1
9,1,0.999,1.0,0.992,0.997,0.997,1


## 09. VotingRegressor 회귀 예측

- VotingRegressor는 여러 회귀 모델의 예측값을 평균내어 최종 예측을 만듦.
- 분류의 Hard/Soft Voting처럼 class를 투표하는 것이 아니라 숫자 예측값을 평균냄.
- 여기서는 Diabetes 회귀 데이터를 사용해 개별 회귀 모델과 VotingRegressor를 비교함.



In [9]:
diabetes = load_diabetes(as_frame=True)

reg_X = diabetes.data
reg_y = diabetes.target

print('회귀 데이터:', reg_X.shape, reg_y.shape)
display(reg_X.head())


회귀 데이터: (442, 10) (442,)


,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641


In [10]:
reg_X_train, reg_X_test, reg_y_train, reg_y_test = train_test_split(
    reg_X,
    reg_y,
    test_size=0.2,
    random_state=42
)

# 스케일링
reg_scaler = StandardScaler()
reg_X_train_scaled = reg_scaler.fit_transform(reg_X_train)
reg_X_test_scaled = reg_scaler.transform(reg_X_test)

# 일반 회귀 모델 생성
lin_reg = LinearRegression()
knn_reg = KNeighborsRegressor(n_neighbors=5)
dt_reg = DecisionTreeRegressor(max_depth=4, random_state=42)

# VotingRegressor는 voting 방식 지정 X
# -> 입력값(X)에 대한 각 모델의 예측값(y_pred)의 평균을 구한다
voting_reg = VotingRegressor(
    estimators=[
        ('linear', lin_reg),
        ('knn', knn_reg),
        ('dt', dt_reg),
    ]
)

reg_models = {
    'linear_regression': lin_reg,
    'knn_regressor': knn_reg,
    'decision_tree_regressor': dt_reg,
    'voting_regressor': voting_reg
}

reg_results = []

# 모델들을 순서대로 학습시키고 RMSE, R2 값 구하기
for name, model in reg_models.items():
    model.fit(reg_X_train_scaled, reg_y_train)

    pred = model.predict(reg_X_test_scaled)

    reg_results.append({
        'model': name,
        'RMSE': root_mean_squared_error(reg_y_test, pred),
        'R2': r2_score(reg_y_test, pred)
    })

reg_result_df = pd.DataFrame(reg_results).sort_values('RMSE')
display(reg_result_df)

,model,RMSE,R2
3,voting_regressor,52.526867,0.479239
0,linear_regression,53.853446,0.452603
1,knn_regressor,55.203713,0.424809
2,decision_tree_regressor,59.740817,0.326375


## 10. Voting 정리

- Hard Voting은 class 예측값을 다수결로 결합함.
- Soft Voting은 class 확률을 평균내어 결합함.
- VotingRegressor는 숫자 예측값을 평균내어 결합함.
- 앙상블은 무조건 성능을 올리는 마법이 아니라, 서로 다른 모델의 장점을 잘 조합할 때 효과가 커짐.

